##  Simple Transport between graphs using FGW

This notebook proposes to compute the assignment of labeled graphs using FGW

In [ ]:
import numpy as np
import os,sys
sys.path.append(os.path.realpath('../lib'))
from graph_sentence_temporalfinal import graph_colors,draw_rel,draw_transp,Graph,wl_labeling,find_thresh,sp_to_adjency
from ot_distances import Fused_Gromov_Wasserstein_distance,Wasserstein_distance
import copy
from data_loader import load_local_data,histog,build_noisy_circular_graph
import matplotlib.pyplot as plt
import networkx as nx
from sentence_transformers import SentenceTransformer
from FGW import fgw_barycenters

First example is a transport between two simple graphs with four nodes

In [ ]:
method='shortest_path_weight'
embedding = SentenceTransformer('all-MiniLM-L6-v2')
print("add sentences")
g1=Graph()
g1.add_sentence("{f: scared}[exhales sharply] No... do {g: Shaking Head No}do NOT {t: Negation} take the left road.",embedding)
g2=Graph()
g2.add_sentence("{f: confident}[exhales softly] No... do {g: Shaking Head No}do NOT {t: Negation} take the left road.",embedding)

In [ ]:
from sentence_transformers import SentenceTransformer

# 1. Load the pre-trained model
# This model is a great balance of speed and accuracy
model = SentenceTransformer('sentence-transformers/bert-base-nli-mean-tokens')

# 2. Your node descriptions
# These are the annotations you have for each node in your graph
node_descriptions = [
    "Use a facial expression happy", 
    "Stand sadly", 
    "Use a ffacial expression sad", 
    "Use a ffacial expression joyful",
    "energetic waving gesture"
]

# 3. Generate Embeddings (The "Zero-Shot" part)
# The model converts each text description into a 384-dimensional vector
embeddings = model.encode(node_descriptions)

# Now you have a matrix of shape (5, 384)
# This matrix is exactly what you pass into the 'emb' part of the FGW distance!

In [ ]:
# train a pca on the embeddings of all tags
import pandas as pd
from sklearn.decomposition import PCA
possible_tags = []
gesture_data = pd.read_csv('..\data\gestures.csv')
for i, row in gesture_data.iterrows():

    possible_tags.append("Gesture "+row['Description'])
intonations = ['thoughtful']
for i in intonations:
    possible_tags.append("Talk with a " + i + " intonation")
facial_expressions=['happy']
for f in facial_expressions:
    possible_tags.append("facial expression " + f)
linguistic_tags=['Negation']
for t in linguistic_tags:
    possible_tags.append( t + "in the text")
X = embeddings = model.encode(possible_tags)
pca = PCA(n_components=2)
pca.fit(X)

In [ ]:
embeddings = pca.transform(embeddings)

In [ ]:
import seaborn as sns
import numpy as np
print("distance happy joyful", np.linalg.norm(embeddings[0]-embeddings[3]))
print("distance happy sad", np.linalg.norm(embeddings[0]-embeddings[1]))
print("distance happy wave", np.linalg.norm(embeddings[0]-embeddings[4]))

In [ ]:
#plt.figure(figsize=(5,4))
vmin=0
vmax=7
draw_rel(g1.nx_graph,draw=False,vmin=-5,vmax=5,with_labels=True)
#draw_rel(g2.nx_graph,draw=False,vmin=-5,vmax=5,with_labels=True,shiftx=5,swipy=True)
plt.title('Two graphs. Color indicates the label')
plt.show()

In [ ]:
sample_text = "Hello {f: smile} how are you [excited] {g: Shaking Head No} and how is {t: Negation} it going {f: neutral} {g: Talking 2}"
gif_path = g1.create_sentence_gif(sample_text)
print(f"Animation successfully created at: {gif_path}")

In [ ]:
alpha=0.5
dfgw=Fused_Gromov_Wasserstein_distance(alpha=alpha,features_metric='sqeuclidean',method=method).graph_d(g1,g2)
dw=Wasserstein_distance(features_metric='sqeuclidean').graph_d(g1,g2)
dgw=Fused_Gromov_Wasserstein_distance(alpha=1,features_metric='sqeuclidean',method=method).graph_d(g1,g2)
print('Wasserstein distance={}, Gromov distance={} \nFused Gromov-Wasserstein distance for alpha {} = {}'.format(dw,dgw,alpha,dfgw))

We see that neither Wasserstein distance on the label, or Gromov distance can discriminate between this two graphs, whereas FGW does not vanish.

We now build more complicated graphs with the following :

We plot the evolution of the FGW distance with respect to the alpha parameter

In [ ]:
alld=[]
x=np.linspace(0,1,10000)
for alpha in x:
    d=Fused_Gromov_Wasserstein_distance(alpha=alpha,features_metric='sqeuclidean',method=method).graph_d(g1,g2)
    alld.append(d)
plt.plot(x,alld)
plt.title('Evolution of FGW dist in wrt alpha \n max={}'.format(x[np.argmax(alld)]))
plt.xlabel('Alpha')
plt.ylabel('FGW dist')
plt.show()

We see that the alpha which leads to best discriminate the graphs is between 0 and 1. We plot the coupling using this alpha :

In [ ]:
fig=plt.figure(figsize=(10,8))
thresh=0.004
gwdist=Fused_Gromov_Wasserstein_distance(alpha=0.5,features_metric='sqeuclidean',method=method)
d=gwdist.graph_d(g1,g2)
plt.title('FGW coupling, dist : '+str(np.round(d,3)),fontsize=15)
draw_transp(g1,g2,gwdist.transp,shiftx=2,shifty=0.5,thresh=thresh,
            swipy=True,swipx=False,with_labels=True,vmin=vmin,vmax=11)
plt.show()

In [ ]:
gwdist.transp

We can also plot the GW coupling

In [ ]:
fig=plt.figure(figsize=(10,8))
thresh=0.004
gwdist=Fused_Gromov_Wasserstein_distance(alpha=1,features_metric='sqeuclidean',method=method)
d=gwdist.graph_d(g1,g2)
plt.title('GW coupling, dist : '+str(np.round(d,3)),fontsize=15)
draw_transp(g1,g2,gwdist.transp,shiftx=2,shifty=0.5,thresh=thresh,
            swipy=False,swipx=False,with_labels=True,vmin=vmin,vmax=11)

plt.show()

In [ ]:
import pandas as pd
data = pd.read_csv('../data_with_text_tag\generated_instructions_with_intention_explanation_full_ability_gender_neutral_gpt.csv')
data['instruction_with_intention'][0]

In [ ]:
def create_graph_from_data(x,embedding):
    graphs=Graph()
    graphs.add_sentence(x,embedding)
    return graphs

In [ ]:

data_high = data[data['ability']=='High']
data_low = data[data['ability']=='Low']
data_medium = data[data['ability']=='Medium']
graphs_high=[create_graph_from_data(x,embedding) for x in data_high.instruction_with_intention]
graphs_low=[create_graph_from_data(x,embedding) for x in data_low.instruction_with_intention]
graphs_medium=[create_graph_from_data(x,embedding) for x in data_medium.instruction_with_intention]

In [ ]:
def from_numpy_array(G, adj_matrix, source_node_offset=0):
        """
        Populates the graph from a numpy adjacency matrix.
        
        Args:
            adj_matrix (np.ndarray): Square matrix where adj_matrix[i, j] is the weight.
            source_node_offset (int): The starting ID for nodes. 
                                      Useful if you are adding this matrix to an existing graph.
        """
        import numpy as np
        
        rows, cols = adj_matrix.shape
        if rows != cols:
            raise ValueError("Adjacency matrix must be square.")

        # 1. Ensure all nodes exist in the graph first
        for i in range(rows):
            node_id = i + source_node_offset
            if not G.nx_graph.has_node(node_id):
                G.nx_graph.add_node(node_id)
        
        # 2. Iterate through the matrix to add edges
        # We use np.nonzero to avoid iterating over thousands of zeros in sparse matrices
        nonzero_indices = np.nonzero(adj_matrix)
        
        for i, j in zip(*nonzero_indices):
            weight = adj_matrix[i, j]
            u = i + source_node_offset
            v = j + source_node_offset
            
            # Add edge with the weight from the matrix
            # You can add additional default attributes here (e.g., type='matrix_imported')
            G.nx_graph.add_edge(u, v, weight=weight)
            
        # 3. Update the internal node counter to prevent ID collisions in future additions
        G.n_nodes = max(G.n_nodes, rows + source_node_offset)
        
        print(f"Successfully imported {len(nonzero_indices[0])} edges from matrix.")


In [ ]:
def createbary(graphs):
    for x in graphs:
        #if x.nx_graph has no nodes of edges, remove it from the list
        if len(x.nodes())==0 or len(x.edges())==0:
            graphs.remove(x)
    Cs=[x.distance_matrix(force_recompute=True,method=method) for x in graphs]
    ps=[np.ones(len(x.nodes()))/len(x.nodes()) for x in graphs]
    Ys=[x.values() for x in graphs]
    lambdas=np.array([np.ones(len(Ys))/len(Ys)]).ravel()
    sizebary=(15,384) # we choose a barycenter with 15 nodes
    init_X=np.zeros((sizebary[0],sizebary[1]))
    D1_high,C1_high,log=fgw_barycenters(sizebary[0],Ys,Cs,ps,lambdas,alpha=0.95,init_X=init_X)
    bary_high = Graph()
    from_numpy_array(bary_high, sp_to_adjency(C1_high, threshinf=0, threshsup=find_thresh(C1_high, sup=100, step=100)[0]))
    for i in range(len(D1_high)):
        bary_high.nx_graph.add_node(i, attr_name=D1_high[i])
    return bary_high

In [ ]:
bary_high=createbary(graphs_high)

In [ ]:
draw_rel(bary_high .nx_graph,draw=False,vmin=-5,vmax=5,with_labels=True)

In [ ]:
bary_medium=createbary(graphs_medium)

In [ ]:
draw_rel(bary_medium.nx_graph,draw=False,vmin=-5,vmax=5,with_labels=True)

In [ ]:
bary_low=createbary(graphs_low)

In [ ]:
draw_rel(bary_low.nx_graph,draw=False,vmin=-5,vmax=5,with_labels=True)

In [ ]:
bary_low.nodes()

In [ ]:
gwdist=Fused_Gromov_Wasserstein_distance(alpha=0.8,features_metric='sqeuclidean',method=method)
d_low_high=gwdist.graph_d(bary_low,bary_high)*10000
d_low_medium=gwdist.graph_d(bary_low,bary_medium)*10000
d_medium_high=gwdist.graph_d(bary_medium,bary_high)*10000

#plot confusion matrix of distances between barycenters
import seaborn as sns
import matplotlib.pyplot as plt
distances = np.array([[0, d_low_medium, d_low_high],
                        [d_low_medium, 0, d_medium_high],
                        [d_low_high, d_medium_high, 0]])
plt.figure(figsize=(8, 6))
sns.heatmap(distances, annot=True, fmt=".2f", cmap="YlGnBu", xticklabels=['Low', 'Medium', 'High'], yticklabels=['Low', 'Medium', 'High'])
plt.title('FGW Distances Between Barycenters')
plt.xlabel('Barycenter')
plt.ylabel('Barycenter')
plt.show()

In [ ]:
alld=[]
x=np.linspace(0,1,10000)
for alpha in x:
    d=Fused_Gromov_Wasserstein_distance(alpha=alpha,features_metric='sqeuclidean',method=method).graph_d(bary_low,bary_high)*10000
    alld.append(d)
plt.plot(x,alld)
plt.title('Evolution of FGW dist in wrt alpha \n max={}'.format(x[np.argmax(alld)]))
plt.xlabel('Alpha')
plt.ylabel('FGW dist')
plt.show()

In [ ]:
alld=[]
x=np.linspace(0,1,10000)
for alpha in x:
    d=Fused_Gromov_Wasserstein_distance(alpha=alpha,features_metric='sqeuclidean',method=method).graph_d(bary_low,bary_medium)*10000
    alld.append(d)
plt.plot(x,alld)
plt.title('Evolution of FGW dist in wrt alpha \n max={}'.format(x[np.argmax(alld)]))
plt.xlabel('Alpha')
plt.ylabel('FGW dist')
plt.show()

In [ ]:
alld=[]
x=np.linspace(0,1,10000)
for alpha in x:
    d=Fused_Gromov_Wasserstein_distance(alpha=alpha,features_metric='sqeuclidean',method=method).graph_d(bary_medium,bary_high)*10000
    alld.append(d)
plt.plot(x,alld)
plt.title('Evolution of FGW dist in wrt alpha \n max={}'.format(x[np.argmax(alld)]))
plt.xlabel('Alpha')
plt.ylabel('FGW dist')
plt.show()